In [3]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/APD_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
print(df.head())
print(df['year'].unique())
print(df.columns)
# Keep only one row per location-year (average if needed)
# Define feature columns
features = ['BSI', 'Forest', 'NBR', 'NDMI', 'NDVI']

# Now group properly
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

             system:index       BSI  Forest       NBR      NDMI      NDVI  \
0  00000000000000061efc_0 -0.308401       1  0.656181  0.342823  0.861842   
1  00000000000000061efc_0 -0.161510       1  0.399157  0.231395  0.498130   
2  00000000000000061efc_0 -0.158983       1  0.400300  0.227913  0.479325   
3  00000000000000061efc_0 -0.191802       1  0.441187  0.265590  0.536595   
4  00000000000000061efd_0 -0.339410       1  0.682075  0.370293  0.885978   

   year                                               .geo  longitude  \
0  2021  {"geodesic":false,"type":"Point","coordinates"...  96.717564   
1  2022  {"geodesic":false,"type":"Point","coordinates"...  96.717564   
2  2023  {"geodesic":false,"type":"Point","coordinates"...  96.717564   
3  2024  {"geodesic":false,"type":"Point","coordinates"...  96.717564   
4  2021  {"geodesic":false,"type":"Point","coordinates"...  96.718463   

    latitude  
0  27.461947  
1  27.461947  
2  27.461947  
3  27.461947  
4  27.461947  
[2021 20

In [5]:
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

# Make sure duplicates are removed
df_unique = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

for (lat, lon), group in df_unique.groupby(['latitude', 'longitude']):
    group = group.sort_values('year')
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)

Raw input shape: (30000, 4, 5)


In [6]:

# ===============================
# 4. Label Creation (BEFORE SCALING)
# ===============================
# NDVI drop = vegetation loss
ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # 2021 - 2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # 2024 - 2021

ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# ===============================
# 5. Train–Test Split (WITH LOCATIONS)
# ===============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
    X_raw, y, locations,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ===============================
# 6. Scaling (NO DATA LEAKAGE)
# ===============================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# ===============================
# 7. LSTM Model
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

model.summary()

# ===============================
# 8. Train Model (Class Weighting)
# ===============================
class_weight = {0: 1, 1: 12}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    class_weight=class_weight,
    verbose=1
)

# ===============================
# 9. Evaluation
# ===============================
from sklearn.metrics import classification_report

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)   # recall-friendly threshold

print(classification_report(y_test, y_pred))

# ===============================
# 10. Save Predictions (CORRECT LOCATIONS)
# ===============================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/APD_Deforestation_Predictions.csv',
    index=False
)

print("Saved:RJ_Deforestation_Predictions.csv")



Label distribution:
No deforestation: 26663
Deforestation: 3337
Train shape: (24000, 4, 5)
Test shape : (6000, 4, 5)
Scaled train shape: (24000, 4, 5)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,033 (78.25 KB)

 Trainable params: 20,033 (78.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.3050 - loss: 1.3373 - precision: 0.1369 - recall: 0.9392 - val_accuracy: 0.8163 - val_loss: 0.4223 - val_precision: 0.3539 - val_recall: 0.7901
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.7647 - loss: 0.8854 - precision: 0.3099 - recall: 0.8811 - val_accuracy: 0.8003 - val_loss: 0.4087 - val_precision: 0.3428 - val_recall: 0.8681
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.7563 - loss: 0.8487 - precision: 0.3027 - recall: 0.8966 - val_accuracy: 0.8165 - val_loss: 0.3687 - val_precision: 0.3656 - val_recall: 0.8846
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.7876 - loss: 0.7434 - precision: 0.3344 - recall: 0.9166 - val_accuracy: 0.8787 - val_loss: 0.2436 - val_precision: 0.4753 - val_recall: 0.8786
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8170 - loss: 0.6529 - precision: 0.3645 - recall: 0.9172 - val_accuracy: 0.8967 - val_loss

In [7]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/APD_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/APD_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/APD_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())


NDVI: {'mean': np.float64(-0.35287144401649334), 'std': 0.04306974354897533, 'q1': np.float64(-0.37871003999999997), 'q3': np.float64(-0.32950643999999996), 'lower_2std': np.float64(-0.439010931114444), 'upper_2std': np.float64(-0.2667319569185427)}
NBR : {'mean': np.float64(-0.22883593591013335), 'std': 0.04145072191460747, 'q1': np.float64(-0.25593222000000004), 'q3': np.float64(-0.20506149750000002), 'lower_2std': np.float64(-0.3117373797393483), 'upper_2std': np.float64(-0.1459344920809184)}
BSI : {'mean': np.float64(0.11812715626906273), 'std': 0.0390508429528547, 'q1': np.float64(0.09634453865018164), 'q3': np.float64(0.1454138951845781), 'lower_2std': np.float64(0.04002547036335333), 'upper_2std': np.float64(0.19622884217477213)}
NDMI: {'mean': np.float64(-0.08707247438255668), 'std': 0.033623522888034045, 'q1': np.float64(-0.10943926750000002), 'q3': np.float64(-0.06815304000000003), 'lower_2std': np.float64(-0.15431952015862477), 'upper_2std': np.float64(-0.019825428606488588)

In [9]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Andhra Pradesh map
# AP roughly: latitude 12.6–19.9, longitude 76.8–84.8
m = folium.Map(location=[16.5, 80.6], zoom_start=7)

# Load CSV
df = pd.read_csv('/content/drive/MyDrive/APD_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}
# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)


cause
Urban/Other    1088
Unknown          88
Name: count, dtype: int64


In [10]:
m.save('/content/drive/MyDrive/apd_adaptive.html')


In [11]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS
# ==============================

df_def = pd.read_csv(
    '/content/drive/MyDrive/APD_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()

print("Total deforestation samples:", len(df_deforest))


# ==============================
# LOAD CAUSE DATA
# ==============================

df_cause = pd.read_csv(
    '/content/drive/MyDrive/APD_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())


# ==============================
# LOAD ARUNACHAL PRADESH DISTRICTS
# ==============================

apd = gpd.read_file('/content/drive/MyDrive/APD.geojson')

# CRS SAFETY
if apd.crs is None:
    apd.set_crs("EPSG:4326", inplace=True)

apd = apd.to_crs("EPSG:4326")

# Add state label (optional)
apd["state"] = "Arunachal Pradesh"

print(apd.head())


# ==============================
# MERGE CAUSE DATA
# ==============================

df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())


# ==============================
# CREATE POINT GEOMETRY
# ==============================

geometry = [
    Point(xy) for xy in zip(
        df_deforest['longitude'],
        df_deforest['latitude']
    )
]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 6000
    latitude  longitude  deforestation
0  27.511355  96.469629              0
1  27.471829  96.452561              0
2  27.468236  96.687022              0
3  27.502372  96.689716              0
4  27.513151  96.862193              1
Total deforestation samples: 1176
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  27.513151  96.862193              1    -0.392592   -0.187995    0.058509   
1  27.479015  96.820870              1    -0.332425   -0.170987    0.053486   
2  27.481710  96.837938              1    -0.324596   -0.230581    0.057207   
3  27.481710  96.479511              1    -0.280212   -0.138197    0.053362   
4  27.461947  96.855905              1          NaN         NaN         NaN   

   NDMI_change        cause  
0    -0.004748  Urban/Other  
1    -0.033093  Urban/Other  
2    -0.066478  Urban/Other  
3    -0.024865  Urban/Other  
4          NaN      Unknown  
  REMARKS_2 Country         State_Name State_Code        

In [12]:
# ==============================
# SPATIAL JOIN (Points → Districts)
# ==============================

gdf_joined = gpd.sjoin(
    gdf_points,
    apd,                # Arunachal Pradesh GeoDataFrame
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['Dist_Name'].isna().sum())

print(gdf_joined['Dist_Name'].value_counts())


# ==============================
# FIX DUPLICATE CAUSE COLUMNS
# ==============================

if 'cause' not in gdf_joined.columns:

    if 'cause_x' in gdf_joined.columns and 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x'].fillna(gdf_joined['cause_y'])

    elif 'cause_x' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x']

    elif 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_y']

# cleanup
gdf_joined.drop(
    columns=[c for c in ['cause_x','cause_y'] if c in gdf_joined.columns],
    inplace=True
)


# ==============================
# DISTRICT SUMMARY — APD
# ==============================

district_summary = (
    gdf_joined
    .groupby('Dist_Name')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)

district_summary.to_csv(
    '/content/drive/MyDrive/APD_District_Wise_Deforestation.csv',
    index=False
)

print("Saved APD district summary")


# ==============================
# DISTRICT × CAUSE SUMMARY
# ==============================

cause_summary = (
    gdf_joined
    .groupby(['Dist_Name', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/APD_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved APD cause summary")

    latitude  longitude  deforestation        cause  \
0  27.513151  96.862193              1  Urban/Other   
1  27.479015  96.820870              1  Urban/Other   
2  27.481710  96.837938              1  Urban/Other   
3  27.481710  96.479511              1  Urban/Other   
4  27.461947  96.855905              1      Unknown   

                    geometry  index_right REMARKS_2 Country  \
0  POINT (96.86219 27.51315)            8      None   India   
1  POINT (96.82087 27.47902)            8      None   India   
2  POINT (96.83794 27.48171)            8      None   India   
3  POINT (96.47951 27.48171)            8      None   India   
4   POINT (96.8559 27.46195)            8      None   India   

          State_Name State_Code  Dist_Name Dist_Code              state  
0  Arunachal Pradesh         12  Changlang       253  Arunachal Pradesh  
1  Arunachal Pradesh         12  Changlang       253  Arunachal Pradesh  
2  Arunachal Pradesh         12  Changlang       253  Arunachal Prad

In [13]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Arunachal Pradesh District Boundaries
# =====================================================
apd = gpd.read_file('/content/drive/MyDrive/APD.geojson')

# CRS safety
if apd.crs is None:
    apd.set_crs(epsg=4326, inplace=True)

apd = apd.to_crs(epsg=4326)
apd["state"] = "Arunachal Pradesh"
gdf_districts = apd.copy()

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/APD_Deforestation_Predictions.csv")

# Ensure afforestation column exists
if 'afforestation' not in df.columns:
    df['afforestation'] = 0  # placeholder if no data

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join (assign districts to points)
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# =====================================================
# STEP 7: Merge Stats into District Map
# =====================================================
gdf_districts = gdf_districts.merge(
    district_total, on="Dist_Name", how="left"
)
gdf_districts = gdf_districts.merge(
    district_deforestation, on="Dist_Name", how="left"
)
gdf_districts = gdf_districts.merge(
    district_afforestation, on="Dist_Name", how="left"
)

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 8: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

# Deforestation
gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

# Afforestation
gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 9: Create Folium Map (center Arunachal Pradesh)
# =====================================================
m = folium.Map(location=[28.2, 94.7], zoom_start=7)

# =====================================================
# STEP 10: Dynamic Choropleth for Deforestation
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Arunachal Pradesh)"
).add_to(m)

# =====================================================
# STEP 11: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 12: Dynamic Arunachal Pradesh Afforestation Alert Popup
# =====================================================
state_total = gdf_districts["total_samples"].sum()
state_def = gdf_districts["deforestation_samples"].sum()
state_aff = gdf_districts["afforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Arunachal Pradesh Afforestation Alert</b><br><br>

Total Deforestation Points: <b>{int(state_def)}</b><br><br>

High Impact Districts:<br>
{top_districts_html}<br>

🌱 Immediate afforestation drives recommended.<br>
Focus on community forestry, biodiversity protection,<br>
and sustainable land management.
"""

folium.Marker(
    location=[28.2, 94.7],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 13: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/apd_forest.html")

print("✅ Arunachal Pradesh map saved successfully with afforestation recommendation!")

/tmp/ipython-input-440/3807169242.py:88: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ Arunachal Pradesh map saved successfully with afforestation recommendation!
